PySpark DataFrame Case Study: Employee Performance Review Analysis
Scenario: You have a dataset containing employee performance reviews over time and want to analyze performance trends, average ratings, and identify high performers. The dataset contains the following columns:
• emp_id: Employee ID
• review_date: Date of the performance review
• department: Department name
• rating: Performance rating (1-5 scale)
• reviewer: Name of the person conducting the review

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, coalesce
from pyspark.sql.window import Window

# Create Spark Session
spark = SparkSession.builder \
    .appName("Employee Performance Review Analysis") \
    .getOrCreate()

# Sample Data
data = [
    (1, '2024-01-10', 'Engineering', 5, 'Nisha'),
    (2, '2024-01-11', 'HR', 4, 'Ayush'),
    (3, '2024-01-12', 'Sales', 3, 'Ananya'),
    (4, '2024-02-01', 'Engineering', 5, 'Bipina'),
    (1, '2024-03-10', 'Engineering', 4, 'Jane'),
    (2, '2024-03-11', 'HR', None, 'Sam')
]

# Column Names
columns = ["emp_id", "review_date", "department", "rating", "reviewer"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Show DataFrame
df.show()

+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|     5|   Nisha|
|     2| 2024-01-11|         HR|     4|   Ayush|
|     3| 2024-01-12|      Sales|     3|  Ananya|
|     4| 2024-02-01|Engineering|     5|  Bipina|
|     1| 2024-03-10|Engineering|     4|    Jane|
|     2| 2024-03-11|         HR|  NULL|     Sam|
+------+-----------+-----------+------+--------+



In [0]:
#Find employees with ratings greater than or equal to 4

df_filtered = df.filter(col("rating") >= 4)

df_filtered.show()

#The filter() function is used to retrieve records where employee ratings are 4 or higher.

+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|     5|   Nisha|
|     2| 2024-01-11|         HR|     4|   Ayush|
|     4| 2024-02-01|Engineering|     5|  Bipina|
|     1| 2024-03-10|Engineering|     4|    Jane|
+------+-----------+-----------+------+--------+



In [0]:
#Replace null ratings with the department average rating.
window_spec = Window.partitionBy("department")

df_filled = df.withColumn(
    "rating",
    coalesce(col("rating"), avg("rating").over(window_spec))
)

df_filled.show()

#here Window.partitionBy() creates department-wise partitions.
#     avg() calculates average rating within each department.
#      coalesce() replaces null values with the calculated average.


+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|   5.0|   Nisha|
|     4| 2024-02-01|Engineering|   5.0|  Bipina|
|     1| 2024-03-10|Engineering|   4.0|    Jane|
|     2| 2024-01-11|         HR|   4.0|   Ayush|
|     2| 2024-03-11|         HR|   4.0|     Sam|
|     3| 2024-01-12|      Sales|   3.0|  Ananya|
+------+-----------+-----------+------+--------+



In [0]:
#Remove duplicate reviews based on employee ID and review date.

df_no_duplicates = df.dropDuplicates(["emp_id", "review_date"])

df_no_duplicates.show()

# Here the dropDuplicates() function removes repeated records from the dataset.



+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|     5|   Nisha|
|     2| 2024-01-11|         HR|     4|   Ayush|
|     3| 2024-01-12|      Sales|     3|  Ananya|
|     4| 2024-02-01|Engineering|     5|  Bipina|
|     1| 2024-03-10|Engineering|     4|    Jane|
|     2| 2024-03-11|         HR|  NULL|     Sam|
+------+-----------+-----------+------+--------+



In [0]:
#Display only selected columns.

df_selected = df.select("emp_id", "department", "rating")

df_selected.show()

#df.select() is used to select specific columns from the DataFrame.

+------+-----------+------+
|emp_id| department|rating|
+------+-----------+------+
|     1|Engineering|     5|
|     2|         HR|     4|
|     3|      Sales|     3|
|     4|Engineering|     5|
|     1|Engineering|     4|
|     2|         HR|  NULL|
+------+-----------+------+



In [0]:
#Calculate average rating for each department.

df_grouped = df.groupBy("department") \
               .agg(avg("rating").alias("avg_rating"))

df_grouped.show()

# groupBy() groups records by department.
# agg() performs aggregation.
# avg() calculates the average rating.

+-----------+-----------------+
| department|       avg_rating|
+-----------+-----------------+
|Engineering|4.666666666666667|
|         HR|              4.0|
|      Sales|              3.0|
+-----------+-----------------+



In [0]:
 # Join employee review data with employee details.
 employee_data = [
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie"),
    (4, "David")
]

employee_columns = ["emp_id", "employee_name"]

df_employees = spark.createDataFrame(employee_data, employee_columns)

df_joined = df.join(df_employees, on="emp_id", how="inner")

df_joined.show()

#The join() function combines employee review data with employee information using emp_id.


+------+-----------+-----------+------+--------+-------------+
|emp_id|review_date| department|rating|reviewer|employee_name|
+------+-----------+-----------+------+--------+-------------+
|     1| 2024-01-10|Engineering|     5|   Nisha|        Alice|
|     2| 2024-01-11|         HR|     4|   Ayush|          Bob|
|     3| 2024-01-12|      Sales|     3|  Ananya|      Charlie|
|     4| 2024-02-01|Engineering|     5|  Bipina|        David|
|     1| 2024-03-10|Engineering|     4|    Jane|        Alice|
|     2| 2024-03-11|         HR|  NULL|     Sam|          Bob|
+------+-----------+-----------+------+--------+-------------+



In [0]:
#Add new review records to the existing dataset.

new_reviews = [
    (5, '2024-04-01', 'Marketing', 5, 'Chris')
]

df_new_reviews = spark.createDataFrame(new_reviews, columns)

df_union = df.union(df_new_reviews)

df_union.show()

#The union() function appends rows from one DataFrame to another.

+------+-----------+-----------+------+--------+
|emp_id|review_date| department|rating|reviewer|
+------+-----------+-----------+------+--------+
|     1| 2024-01-10|Engineering|     5|   Nisha|
|     2| 2024-01-11|         HR|     4|   Ayush|
|     3| 2024-01-12|      Sales|     3|  Ananya|
|     4| 2024-02-01|Engineering|     5|  Bipina|
|     1| 2024-03-10|Engineering|     4|    Jane|
|     2| 2024-03-11|         HR|  NULL|     Sam|
|     5| 2024-04-01|  Marketing|     5|   Chris|
+------+-----------+-----------+------+--------+



In [0]:
#Find average rating for each employee using SQL.
df.createOrReplaceTempView("performance_reviews")

sql_result = spark.sql("""
    SELECT
        emp_id,
        AVG(rating) AS avg_rating
    FROM performance_reviews
    GROUP BY emp_id
""")
sql_result.show()

#createOrReplaceTempView() creates a temporary SQL table.
#Spark SQL is used for querying the DataFrame.

+------+----------+
|emp_id|avg_rating|
+------+----------+
|     1|       4.5|
|     2|       4.0|
|     3|       3.0|
|     4|       5.0|
+------+----------+



In [0]:
#Calculate cumulative average rating for each employee over time.
window_spec = Window.partitionBy("emp_id") \
                    .orderBy("review_date")

df_with_cumulative_avg = df.withColumn(
    "cumulative_avg",
    avg("rating").over(window_spec)
)

df_with_cumulative_avg.show()

#Window functions perform calculations across rows related to the current row without collapsing the dataset.

+------+-----------+-----------+------+--------+--------------+
|emp_id|review_date| department|rating|reviewer|cumulative_avg|
+------+-----------+-----------+------+--------+--------------+
|     1| 2024-01-10|Engineering|     5|   Nisha|           5.0|
|     1| 2024-03-10|Engineering|     4|    Jane|           4.5|
|     2| 2024-01-11|         HR|     4|   Ayush|           4.0|
|     2| 2024-03-11|         HR|  NULL|     Sam|           4.0|
|     3| 2024-01-12|      Sales|     3|  Ananya|           3.0|
|     4| 2024-02-01|Engineering|     5|  Bipina|           5.0|
+------+-----------+-----------+------+--------+--------------+

